In [1]:
from pathlib import Path

import polars as pl

from ncbi_dataset_builder import BuilderConfig, DatasetBuilder, ResourceSpec

In [2]:
builder = DatasetBuilder(
    BuilderConfig(
        workspace=Path("/workspace-SR003.nfs2/estsoi/test_dataset"),
        email="e.tsoi2@g.nsu.ru",
        max_workers=40,
        total_threads=100,
        ncbi_api_key='e25ae9954b500975211ef84638c26683e307'
    )
)

In [3]:
catalog = builder.fetch_runs(
'''
    "ATAC-seq"[Strategy]
        AND "Homo sapiens"[Organism]
        AND (
        "AsPC-1"[All Fields] OR "AsPC1"[All Fields] OR "ASPC-1"[All Fields]
        OR "PANC-1"[All Fields] OR "PANC1"[All Fields]
        OR "HEK293T"[All Fields] OR "HEK-293T"[All Fields] OR "HEK 293T"[All Fields]
        OR "Huh-7"[All Fields] OR "Huh7"[All Fields]
        OR "SK-HEP-1"[All Fields] OR "SK-HEP1"[All Fields] OR "SK Hep1"[All Fields]
        OR "ARPE-19"[All Fields] OR "ARPE19"[All Fields]
        OR "HepG2"[All Fields] OR "Hep-G2"[All Fields]
        OR "K562"[All Fields] OR "K-562"[All Fields]
        )
'''
)

SRA RunInfo: 8,518 runs loaded from cache; 0 runs require work


In [4]:
# Polars expressions are unrestricted.
catalog = catalog.filter(
    (pl.col("LibraryLayout") == "PAIRED")
    & (pl.col("spots").cast(pl.Int64) >= 50_000_000)
)

In [5]:
metadata = builder.enrich_metadata(catalog, refresh=False)

Sample descriptions: 0 samples loaded from cache; 664 samples require work
Normalized metadata cache miss: /workspace-SR003.nfs2/estsoi/test_dataset/metadata_cache/bundles/064b69ed388ce8bb7e9a32ca4706887e6be8afcbcebe4383556caec4fb61a41a.json
Resolve sra accessions: 0 request batches loaded from raw cache; 16 request batches will be fetched from NCBI


Resolve sra accessions:   0%|          | 0/16 [00:00<?, ?batches/s]

SRA package XML: 0 request batches loaded from raw cache; 8 request batches will be fetched from NCBI


Fetch and parse SRA package XML:   0%|          | 0/8 [00:00<?, ?batches/s]

Resolve biosample accessions: 0 request batches loaded from raw cache; 14 request batches will be fetched from NCBI


...

Save complete metadata JSON:   0%|          | 0/1 [00:00<?, ?file/s]

Save normalized tables:   0%|          | 0/7 [00:00<?, ?tables/s]

Build training descriptions:   0%|          | 0/664 [00:00<?, ?samples/s]

Save sample descriptions:   0%|          | 0/664 [00:00<?, ?samples/s]

Sample description files: 660 unchanged; 4 written


In [6]:
catalog_with_metadata = metadata.attach_to_runs(catalog)

In [7]:
catalog_with_metadata.frame

Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,Experiment,LibraryName,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,InsertSize,InsertDev,Platform,Model,SRAStudy,BioProject,Study_Pubmed_id,ProjectID,Sample,BioSample,SampleType,TaxID,ScientificName,SampleName,g1k_pop_code,source,g1k_analysis_group,Subject_ID,Sex,Disease,Tumor,Affection_Status,Analyte_Type,Histological_Type,Body_Site,CenterName,Submission,dbgap_study_accession,Consent,RunHash,ReadHash,sra_run_title,sra_run_spots,sra_run_bases,sra_run_size_bytes,sra_run_published_at,sra_experiment_title,sra_library_strategy,sra_library_source,sra_library_selection,sra_library_layout,sra_library_construction_protocol,sra_instrument_model,sra_study_accession,sra_study_title,sra_study_abstract,sra_bioproject,sra_sample_title,sra_sample_attributes_json,biosample_title,biosample_organism,biosample_attributes_json
str,datetime[μs],datetime[μs],i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str,i64,f64,str,str,str,str,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ERR15806034""",2026-09-02 19:48:22,2026-09-02 22:38:59,76482256,22944676800,76482256,300,7210,null,"""https://sra-downloadb.be-md.nc…","""ERX15205675""","""FixDMSO_repA_p""","""ATAC-seq""","""PCR""","""GENOMIC""","""PAIRED""",0,0.0,"""ILLUMINA""","""Illumina NovaSeq 6000""","""ERP183325""","""PRJEB101917""",null,1523172,"""ERS27167293""","""SAMEA120466607""","""simple""",9606,"""Homo sapiens""","""SAMEA120466607""",null,null,null,null,null,null,"""no""",null,null,null,null,"""Novogene""","""ERA35098322""",null,"""public""","""B0B2AADE5C8AD6A57D45C8D5597CEB…","""7BD275F440D98846EA5E98D900E5F9…","""Illumina NovaSeq 6000 paired e…",76482256,22944676800,7560980568,"""2026-09-02 19:48:22""","""snRNA Orchestrates ATM Activat…","""ATAC-seq""","""GENOMIC""","""PCR""","""PAIRED""","""DIvA cells were treated with I…","""Illumina NovaSeq 6000""","""ERP183325""","""snRNA Orchestrates ATM Activat…","""Genomic integrity within trans…","""PRJEB101917""","""FixDMSO_repA""","""{""ENA first public"": ""2026-09-…","""FixDMSO_repA""","""Homo sapiens""","""{""ENA first public"": ""2026-09-…"
"""SRR30136592""",2026-08-05 17:25:29,2024-08-05 21:54:22,102634917,9991926371,102634917,97,3310,null,"""https://sra-downloadb.be-md.nc…","""SRX25605821""","""GSM8441658""","""ATAC-seq""","""other""","""GENOMIC""","""PAIRED""",0,0.0,"""ILLUMINA""","""Illumina NovaSeq 6000""","""SRP524294""","""PRJNA1144449""",null,1144449,"""SRS22254478""","""SAMN43038842""","""simple""",9606,"""Homo sapiens""","""GSM8441658""",null,null,null,null,null,null,"""no""",null,null,null,null,"""STRUAN GRANT AND ANDREW WELLS,…","""SRA1940580""",null,"""public""","""836C851B5CB2F7B314D639EBA49B94…","""64DFEC88532C88EB6228607411C964…",null,102634917,9991926371,3471481601,"""2026-08-05 17:25:29""","""GSM8441658: iPSC_Hepatocytes_2…","""ATAC-seq""","""GENOMIC""","""other""","""PAIRED""","""A total of 50,000 to 100,000 c…","""Illumina NovaSeq 6000""","""SRP524294""","""GEO accession GSE274004 is cur…","""If GEO accession GSE274004 has…","""PRJNA1144449""","""iPSC_Hepatocytes_233""","""{""cell type"": ""iPSC_derived_he…","""iPSC_Hepatocytes_233""","""Homo sapiens""","""{""cell_type"": ""iPSC_derived_he…"
"""SRR30136593""",2026-08-05 17:25:29,2024-08-05 22:01:47,129663005,12100641317,129663005,93,4194,null,"""https://sra-downloadb.be-md.nc…","""SRX25605820""","""GSM8441659""","""ATAC-seq""","""other""","""GENOMIC""","""PAIRED""",0,0.0,"""ILLUMINA""","""Illumina NovaSeq 6000""","""SRP524294""","""PRJNA1144449""",null,1144449,"""SRS22254477""","""SAMN43038841""","""simple""",9606,"""Homo sapiens""","""GSM8441659""",null,null,null,null,null,null,"""no""",null,null,null,null,"""STRUAN GRANT AND ANDREW WELLS,…","""SRA1940580""",null,"""public""","""B517DDA733DB9F7D8F8A97C24BA1FB…","""244A6E70

In [12]:
plan = builder.plan(
    catalog_with_metadata,
    group_by="experiment",
    resources=ResourceSpec(threads=100, memory_gb=100, time_limit="24:00:00"),
    max_batch_gb=200,
    max_batch_units=10,
)

Plan dataset from 770 catalog rows
Plan complete: 706 tasks in 76 batches


In [14]:
report = builder.build(plan, 'ncbi_dataset_builder.processing.atac:process_atac')

Genome references for 706 selected tasks in 76 batches: 1 genomes loaded from cache; 0 genomes require work
Batch 0 staging started: 2 units; 199.289 GB estimated


KeyboardInterrupt: 